In [4]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")           
warnings.filterwarnings("ignore")

eda_lines = []

In [2]:
CSV_PATH   = "../dataset/Train_data.csv"

df = pd.read_csv(CSV_PATH)
 
print(f"Shape          : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns        : {list(df.columns)}\n")

Shape          : 1200 rows × 13 columns
Columns        : ['id', 'journal_text', 'ambience_type', 'duration_min', 'sleep_hours', 'energy_level', 'stress_level', 'time_of_day', 'previous_day_mood', 'face_emotion_hint', 'reflection_quality', 'emotional_state', 'intensity']



In [3]:
# Missing value analysis

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"count": missing, "pct": missing_pct})
missing_df = missing_df[missing_df["count"] > 0]

In [5]:
if missing_df.empty:
    print("  No missing values found.")
    eda_lines.append("Missing values: none\n")
else:
    print(missing_df.to_string())
    eda_lines.append(f"Missing values:\n{missing_df.to_string()}\n")

                   count    pct
sleep_hours            7   0.58
previous_day_mood     15   1.25
face_emotion_hint    123  10.25


In [8]:
# Emotional State Analysis

state_counts=df["emotional_state"].value_counts()

majority_pct=(state_counts.iloc[0]*100)/len(df)
print(f"\nEmotional State Distribution:\n{state_counts}\n")
print(f"Majority class percentage: {majority_pct:.2f}%")
eda_lines.append(f"Emotional state distribution:\n{state_counts.to_string()}\n")
eda_lines.append(f"Majority class percentage: {majority_pct:.2f}%\n")


Emotional State Distribution:
emotional_state
calm           216
restless       209
neutral        201
focused        193
mixed          191
overwhelmed    190
Name: count, dtype: int64

Majority class percentage: 18.00%


In [9]:
if majority_pct > 50:
    msg = (f"  ⚠  Class imbalance: '{state_counts.index[0]}' makes up "
           f"{majority_pct:.1f}% of data. Consider class_weight='balanced'.")
    print(msg)
    eda_lines.append(msg + "\n")

In [12]:
# Intensity Distribution

intensity_counts=df["intensity"].value_counts()
print(f"\nIntensity Distribution:\n{intensity_counts}\n")
majority_intensity_pct=(intensity_counts.iloc[0]*100)/len(df)
print(f"Majority intensity percentage: {majority_intensity_pct:.2f}%")
eda_lines.append(f"Intensity distribution:\n{intensity_counts.to_string()}\n")
eda_lines.append(f"Majority intensity percentage: {majority_intensity_pct:.2f}%\n")


Intensity Distribution:
intensity
4    277
3    240
5    229
2    228
1    226
Name: count, dtype: int64

Majority intensity percentage: 23.08%


In [ ]:
# Numerical Feature Statistics
num_cols = ["duration_min", "sleep_hours", "energy_level", "stress_level"]
num_stats = df[num_cols].describe()
print(f"\nNumerical Feature Statistics:\n{num_stats.to_string()}\n")
eda_lines.append(f"\nNumeric stats:\n{num_stats.to_string()}\n")


Numerical Feature Statistics:
       duration_min  sleep_hours  energy_level  stress_level
count   1200.000000  1193.000000   1200.000000   1200.000000
mean      15.861667     5.989522      3.016667      3.026667
std        7.671369     1.500732      1.381296      1.401520
min        3.000000     3.500000      1.000000      1.000000
25%       10.000000     5.000000      2.000000      2.000000
50%       15.000000     6.000000      3.000000      3.000000
75%       20.000000     7.000000      4.000000      4.000000
max       35.000000     8.500000      5.000000      5.000000



In [15]:
# Categorical Feature Analysis
cat_cols = [
    "ambience_type", "time_of_day", "previous_day_mood",
    "face_emotion_hint", "reflection_quality"
]
print("\n[2e] Categorical feature unique values:")
for col in cat_cols:
    vals = sorted(df[col].dropna().unique().tolist())
    print(f"  {col:25s}: {vals}")
    eda_lines.append(f"{col}: {vals}\n")


[2e] Categorical feature unique values:
  ambience_type            : ['cafe', 'forest', 'mountain', 'ocean', 'rain']
  time_of_day              : ['afternoon', 'early_morning', 'evening', 'morning', 'night']
  previous_day_mood        : ['calm', 'focused', 'mixed', 'neutral', 'overwhelmed', 'restless']
  face_emotion_hint        : ['calm_face', 'happy_face', 'neutral_face', 'none', 'tense_face', 'tired_face']
  reflection_quality       : ['clear', 'conflicted', 'vague']


In [ ]:
df["text_word_count"] = df["journal_text"].fillna("").apply(
    lambda x: len(str(x).split())
)
print(f"\n[2f] Journal text word count:")
print(f"  mean={df['text_word_count'].mean():.1f}  "
      f"min={df['text_word_count'].min()}  "
      f"max={df['text_word_count'].max()}")
short_text_rows = df[df["text_word_count"] <= 5]
print(f"  Rows with ≤5 words (very short text): {len(short_text_rows)}")
eda_lines.append(
    f"\nText word count — mean:{df['text_word_count'].mean():.1f} "
    f"min:{df['text_word_count'].min()} max:{df['text_word_count'].max()}\n"
    f"Very short text rows (<=5 words): {len(short_text_rows)}\n"
)
  # Remove the duplicate text word count line


[2f] Journal text word count:
  mean=10.9  min=2  max=32
  Rows with ≤5 words (very short text): 232


In [27]:
eda_lines.remove(eda_lines[-2])

In [28]:
with open("eda_report.txt", "w") as f:
    f.write("EDA REPORT\n" + "=" * 60 + "\n\n")
    f.writelines(eda_lines)
print("\nEDA report saved → eda_report.txt ✓")


EDA report saved → eda_report.txt ✓


In [26]:
print(str(eda_lines[-2]))


Text word count — mean:10.9 min:2 max:32
Very short text rows (≤5 words): 232



In [30]:
ORDINAL_COLS = ["time_of_day", "reflection_quality", "energy_level", "stress_level"]

for col in ORDINAL_COLS:
    print(f"\n[2g] Ordinal column '{col}' unique values:")
    vals = sorted(df[col].dropna().unique().tolist())
    print(f"  {vals}")


[2g] Ordinal column 'time_of_day' unique values:
  ['afternoon', 'early_morning', 'evening', 'morning', 'night']

[2g] Ordinal column 'reflection_quality' unique values:
  ['clear', 'conflicted', 'vague']

[2g] Ordinal column 'energy_level' unique values:
  [1, 2, 3, 4, 5]

[2g] Ordinal column 'stress_level' unique values:
  [1, 2, 3, 4, 5]
